In [11]:
import sys
import json
import random
import csv
import pandas as pd
import numpy as np
import os
from pathlib import Path
from sklearn.model_selection import train_test_split


# Add the project root to the Python path -- It has to to be done before everything
# 必须先把sys的目录改成项目根目录，否则后续的import会找不到项目内模块[例如config.py]
project_root = Path.cwd().parent.parent if Path.cwd().name == 'data' else Path.cwd().parent
sys.path.insert(0, str(project_root))

# Now import from project modules
from src.utils.filesUtil import save_json, save_csv, read_tar
from src.config import (
    RANDOM_SEED, NUM_CLASSES,
    TRAIN_IMG_DIR, TRAIN_ANNOTATION_FILE,
    TEST_IMG_DIR, TEST_ANNOTATION_FILE,
    NUM_TRAIN_PER_CLASS, NUM_VAL_PER_CLASS,
    DATA_SPLITS_DIR
)

import tarfile
import ijson

In [ ]:
def parse_inat_tar_stream_OLD_ERROR(tar_gz_path, img_root_dir):
    """
    错误流式解析
    内存占用在读取的时候有效且恒定，但会在进入具体处理的时候导致内存飙升
    经过排查，因为在流处理的时候我是先加回去再输出的
    images_list.append(...)
    anno_list.append(...)
    所以导致了这个问题，下面有修复的版本
    """
    json_name = None
    with tarfile.open(tar_gz_path, "r:gz") as tar:
        for member in tar:
            if (
                member.isfile()
                and member.name.endswith(".json")
                and not member.name.startswith("/")
                and ".." not in member.name
            ):
                json_name = member.name
                break
    if not json_name:
        raise FileNotFoundError("JSON not found")

    print("analyzing JSON...")
    images_list = []
    anno_list = []
    cat_list = []

    curr_section = None
    curr_item = None
    curr_key = None

    with read_tar(tar_gz_path, json_name) as f:
        for prefix, event, value in ijson.parse(f):
            if prefix == "images.item" and event == "start_map":
                curr_section = "images"
                curr_item = {}
                curr_key = None
                continue
            if prefix == "annotations.item" and event == "start_map":
                curr_section = "annotations"
                curr_item = {}
                curr_key = None
                continue
            if prefix == "categories.item" and event == "start_map":
                curr_section = "categories"
                curr_item = {}
                curr_key = None
                continue

            if curr_item is not None and event == "map_key":
                curr_key = value
                continue

            if curr_item is not None and event in {"string", "number", "boolean", "null"}:
                curr_item[curr_key] = value
                continue

            if curr_item is not None and event == "end_map":
                if curr_section == "images":
                    images_list.append({
                        "image_id": curr_item.get("id"),
                        "file_name": curr_item.get("file_name")
                    })
                elif curr_section == "annotations":
                    anno_list.append({
                        "image_id": curr_item.get("image_id"),
                        "category_id": curr_item.get("category_id")
                    })
                elif curr_section == "categories":
                    cat_list.append({
                        "category_id": curr_item.get("id"),
                        "category_name": curr_item.get("name"),
                        "common_name": curr_item.get("common_name", ""),
                        "kingdom": curr_item.get("kingdom", ""),
                        "supercategory": curr_item.get("supercategory", "")
                    })
                curr_section = None
                curr_item = None
                curr_key = None

    images_df = pd.DataFrame(images_list)
    images_df["file_path"] = images_df["file_name"].apply(
        lambda x: str(Path(img_root_dir) / x)
    )

    anno_df = pd.DataFrame(anno_list)
    cat_df = pd.DataFrame(cat_list)

    print("merging dataframes...")
    merged = anno_df.merge(images_df, on="image_id", how="left")
    merged = merged.merge(cat_df, on="category_id", how="left")

    print(f"parsing complete: {len(merged)} images, {merged['category_id'].nunique()} species")
    return merged

In [13]:
def parse_inat_tar_stream(tar_gz_path, img_root_dir, output_csv_path=None):
    """
    逐步解析 tar 包里的 JSON，并把结果直接写到 CSV。

    ---
    不强制要求写入output_csv_path，会自动生成CSV输出逻辑

    ---

    理论上正确的流式方案
    现在第一遍只收集 images 和 categories 索引，第二遍逐条处理 annotations 并写出结果，
    所以不再需要把所有记录堆成一个超大的 DataFrame，这或许可以在处理图片时参考并节省内存。
    """

    # Convert "input paths"(might be strings) to Path objects
    tar_gz_path = Path(tar_gz_path)
    img_root_dir = Path(img_root_dir)

    # Provide default CSV output path
    if output_csv_path is None:
        source_name = tar_gz_path.name
        if source_name.endswith(".tar.gz"):
            source_name = source_name[:-7]
        else:
            source_name = tar_gz_path.stem
        output_csv_path = Path(DATA_SPLITS_DIR) / f"{source_name}_stream.csv"
        # provide default output path, not hardcoded
    else:
        output_csv_path = Path(output_csv_path)
    print(f"Tar: {tar_gz_path.name}")

    # find the JSON file inside the tar.gz
    json_name = None
    with tarfile.open(tar_gz_path, "r:gz") as tar:
        for member in tar:
            if (
                member.isfile()
                and member.name.endswith(".json")
                and not member.name.startswith("/")
                and ".." not in member.name
            ):
                json_name = member.name
                break
    if not json_name:
        raise FileNotFoundError("未找到JSON文件")
    print("01: found image/category...")

    # Build maps for images and categories
    images_map = {}
    categories_map = {}
    curr_section = None
    curr_item = None
    curr_key = None

    # Read the JSON file in a streaming manner using ijson
    # I had to use this, otherwise my PC will be crushed when I deal with the 42GB pic-files later.
    '''
    ijson.parse(f):
    它不会把 JSON 转换成一个超级巨大的对象把你电脑干碎，而是像传送带似的一个个往外丢，这样就可以减轻单次压力

    例如读到 {"id": 1}，它会依次吐出：
    (start_map),
    (map_key, "id"),
    (number, 1),
    (end_map)
    '''
    with read_tar(tar_gz_path, json_name) as f:
        for prefix, event, value in ijson.parse(f):
            # When it drop "images.item" and a "{", the curr_section will be set to "images" and write the stuff in a dict
            if prefix == "images.item" and event == "start_map":
                curr_section = "images"
                curr_item = {}
                curr_key = None
                continue
            if prefix == "categories.item" and event == "start_map":
                curr_section = "categories"
                curr_item = {}
                curr_key = None
                continue

            if curr_item is not None and event == "map_key":
                curr_key = value
                continue

            if curr_item is not None and event in {"string", "number", "boolean", "null"}:
                curr_item[curr_key] = value
                continue

            # process the collected data, after reaching the "}"
            if curr_item is not None and event == "end_map":
                if curr_section == "images":
                    image_id = curr_item.get("id")
                    file_name = curr_item.get("file_name")
                    if image_id is not None and file_name is not None:
                        images_map[image_id] = file_name
                elif curr_section == "categories":
                    category_id = curr_item.get("id")
                    if category_id is not None:
                        categories_map[category_id] = {
                            "category_name": curr_item.get("name"),
                            "common_name": curr_item.get("common_name", ""),
                            "kingdom": curr_item.get("kingdom", ""),
                            "supercategory": curr_item.get("supercategory", "")
                        }
                curr_section = None
                curr_item = None
                curr_key = None

    print(f"02: Search complete: {len(images_map)} images, {len(categories_map)} species")
    print("03: Writing annotation results...")
    output_csv_path.parent.mkdir(parents=True, exist_ok=True)

    # Write the results to CSV, tried deal with ijson
    written_rows = 0
    missing_images = 0
    missing_categories = 0

    fieldnames = [
        "image_id",
        "file_name",
        "file_path",
        "category_id",
        "category_name",
        "common_name",
        "kingdom",
        "supercategory",
    ]

    with open(output_csv_path, "w", newline="", encoding="utf-8") as csv_file:
        writer = csv.DictWriter(csv_file, fieldnames=fieldnames)
        writer.writeheader()

        with read_tar(tar_gz_path, json_name) as f:
            for annotation in ijson.items(f, "annotations.item"):
                image_id = annotation.get("image_id")
                category_id = annotation.get("category_id")

                file_name = images_map.get(image_id)
                if file_name is None:
                    missing_images += 1
                    continue

                category = categories_map.get(category_id)
                if category is None:
                    missing_categories += 1
                    continue

                writer.writerow({
                    "image_id": image_id,
                    "file_name": file_name,
                    "file_path": str(img_root_dir / file_name),
                    "category_id": category_id,
                    "category_name": category.get("category_name"),
                    "common_name": category.get("common_name", ""),
                    "kingdom": category.get("kingdom", ""),
                    "supercategory": category.get("supercategory", ""),
                })
                written_rows += 1

    print(
        f"Final: provide records: {written_rows} "
        f"Lost image {missing_images}, missing category {missing_categories}"
    )
    print(f"{output_csv_path}")
    return output_csv_path

In [ ]:
parse_inat_tar_stream(
    tar_gz_path=TRAIN_ANNOTATION_FILE,
    img_root_dir=TRAIN_IMG_DIR
)

In [ ]:
"""
    解析iNaturalist2021数据集的json文件
    提取出image_id, file_name, category_id, category_name, supercategory
    
    返回：DataFrame [image_id, file_name, category_id, category_name, supercategory]
"""

 就在这时，我发现我好像不用这么干...我直接在提取500数值的时候同步在提取时就构建list然后把他们扔出来不就行了？